[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/084.%20The%20Multi-Facility%20Location%20-%20p-Center%20Problem/P84-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 84. The Multi-Facility Location: p-Center Problem

## Tier 4: Ensemble Learning

### Purpose and scope

This notebook uses supervised ensemble learning to rank candidate facility locations for the p-center problem. It first generates small problem instances and obtains reference solutions by exhaustive enumeration. It then trains three models on facility-level examples and combines their selection scores with a weighted average. For a new instance, the solver selects the \(p\) candidates with the highest scores and evaluates the resulting set using the p-center objective.

**Inputs** are demand-point coordinates, candidate-facility coordinates, and the number \(p\) of facilities to open. **Outputs** are a set of \(p\) selected candidates, its maximum service distance, and a score for each candidate.

### Model assumptions

- Distances are Euclidean distances in a two-dimensional plane. Coordinates are assumed to use the same scale and units.
- Every demand point is served by its nearest selected facility.
- Exactly \(p\) facilities must be selected from the supplied candidates; candidate locations are fixed.
- Training instances are generated from the same broad coordinate domain as the example. The learned scores may be unreliable when deployment instances differ substantially from that training distribution.
- The training label is whether a candidate belongs to one optimal facility set returned by exhaustive enumeration. If several sets tie, enumeration returns one of them; other equally optimal sets are not separately labeled.
- The model does not represent facility-opening costs, capacity limits, road networks, uncertainty in demand, or other operational constraints.

### The p-center objective

The goal is to place \(p\) facilities so that the farthest demand point is as close as possible to its nearest selected facility. Let \(I\) be the set of demand points, \(J\) the set of candidate facilities, and \(d_{ij}\) the Euclidean distance from demand point \(i\) to candidate \(j\). For a selected set \(S \subseteq J\), with \\(|S|=p\\), its service radius is

$$R(S)=\max_{i\in I}\min_{j\in S}d_{ij}.$$

The optimization problem is

$$\min_{S\subseteq J,\,|S|=p} R(S).$$

Here, \(R(S)\) is the worst-case nearest-facility distance. The code computes it in `PCenterInstance.evaluate_solution`. For the small training instances, `get_optimal_solution` enumerates all \\(\binom{|J|}{p}\\) candidate sets and returns one with minimum radius. Its work is proportional to \\(\binom{|J|}{p}|I|p\\), making it suitable for generating labels on these small instances, but not necessarily for large ones.

### How the supervised ensemble works

Each candidate facility becomes one training row. The row contains instance-wide spatial and distance summaries together with features specific to that candidate. Its binary target is 1 if the candidate appears in the enumerated optimal set and 0 otherwise. Gradient Boosting and a neural network regress this target to obtain scores; a Random Forest classifier estimates the probability of the positive class. The three outputs are combined using the fixed weights in `model_weights`.

For candidate \(j\), the combined score is

$$s_j=0.4\,s_j^{\mathrm{GB}}+0.3\,s_j^{\mathrm{RF}}+0.3\,s_j^{\mathrm{NN}}.$$

The scores rank candidates; selecting the top \(p\) produces a feasible set, which is then evaluated with the original p-center objective. The ensemble is a prediction procedure, not an exact optimization method: its selected set is not guaranteed to be optimal. The exhaustive solution is used as a training reference and as a quality benchmark for the small examples.

### What to examine

- The validation accuracy of each model on held-out problem instances. These are facility-level classification diagnostics, not guarantees of solution quality.
- The candidate scores, selected set, and its maximum service distance.
- The objective gap between the ensemble solution and the exhaustive optimum. A zero gap means that the objective values match, even if tied optimal facility sets differ.
- Performance across independently generated small test instances and the Random Forest's fitted feature-importance values. Feature importance describes the fitted forest; it is not a causal explanation.

### Concrete example

The example uses five demand points, four candidate facilities, and \(p=2\). The cells below compute and display all results when the notebook is run; no numerical prediction is assumed in advance.

In [ ]:
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, List, Optional, Set, Tuple
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
random.seed(42)
np.random.seed(42)
plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class PCenterInstance:
    """Represent a p-center instance with fixed demand and candidate locations."""

    demand_points: List[Tuple[float, float]]
    facility_locations: List[Tuple[float, float]]
    p: int

    def __post_init__(self) -> None:
        if not self.demand_points:
            raise ValueError("At least one demand point is required.")
        if not self.facility_locations:
            raise ValueError("At least one candidate facility is required.")
        if not 1 <= self.p <= len(self.facility_locations):
            raise ValueError("p must be between 1 and the number of candidates.")
        if any(len(point) != 2 for point in self.demand_points + self.facility_locations):
            raise ValueError("Every location must have exactly two coordinates.")

    def compute_distance_matrix(self) -> np.ndarray:
        """Return Euclidean distances, with demand points in rows and candidates in columns."""
        demand = np.asarray(self.demand_points, dtype=float)
        facilities = np.asarray(self.facility_locations, dtype=float)
        differences = demand[:, np.newaxis, :] - facilities[np.newaxis, :, :]
        return np.linalg.norm(differences, axis=2)

    def evaluate_solution(self, facility_set: Set[int]) -> float:
        """Return the maximum nearest-selected-facility distance for a feasible set."""
        if len(facility_set) != self.p:
            raise ValueError("A solution must select exactly p facilities.")
        if any(index < 0 or index >= len(self.facility_locations) for index in facility_set):
            raise ValueError("Facility indices must refer to available candidates.")

        distances = self.compute_distance_matrix()
        selected_distances = distances[:, sorted(facility_set)]
        nearest_distances = np.min(selected_distances, axis=1)
        return float(np.max(nearest_distances))

    def get_optimal_solution(self) -> Tuple[Set[int], float]:
        """Find an optimal set by enumerating every p-candidate combination."""
        best_solution: Optional[Set[int]] = None
        best_objective = float("inf")

        for combination in combinations(range(len(self.facility_locations)), self.p):
            facility_set = set(combination)
            objective = self.evaluate_solution(facility_set)
            if objective < best_objective:
                best_solution = facility_set
                best_objective = objective

        if best_solution is None:
            raise RuntimeError("No feasible facility set was generated.")
        return best_solution, best_objective

In [ ]:
class FeatureExtractor:
    """Build a fixed-length feature vector for one candidate in an instance."""

    def __init__(self) -> None:
        self.feature_names = [
            "n_demand", "n_facilities", "p", "demand_facility_ratio", "selection_ratio",
            "mean_distance", "std_distance", "min_distance", "max_distance", "median_distance",
            "mean_demand_nearest", "std_demand_nearest", "max_demand_nearest",
            "mean_facility_coverage", "std_facility_coverage", "min_facility_coverage",
            "demand_spread", "facility_spread", "center_separation",
            "mean_facility_distance", "std_facility_distance", "min_facility_distance",
            "candidate_mean_demand_distance", "candidate_std_demand_distance",
            "candidate_max_demand_distance", "candidate_mean_peer_distance"
        ]

    def extract_features(self, instance: PCenterInstance, facility_idx: int) -> np.ndarray:
        """Combine instance summaries with distance statistics for one candidate."""
        if not 0 <= facility_idx < len(instance.facility_locations):
            raise IndexError("facility_idx must refer to an available candidate.")

        distances = instance.compute_distance_matrix()
        demand_coords = np.asarray(instance.demand_points, dtype=float)
        facility_coords = np.asarray(instance.facility_locations, dtype=float)
        demand_center = np.mean(demand_coords, axis=0)
        facility_center = np.mean(facility_coords, axis=0)
        flat_distances = distances.ravel()
        demand_nearest = np.min(distances, axis=1)
        facility_coverage = np.min(distances, axis=0)

        if len(facility_coords) > 1:
            pairwise = np.linalg.norm(
                facility_coords[:, np.newaxis, :] - facility_coords[np.newaxis, :, :], axis=2
            )
            upper_triangle = pairwise[np.triu_indices(len(facility_coords), k=1)]
            pairwise_features = [
                float(np.mean(upper_triangle)),
                float(np.std(upper_triangle)),
                float(np.min(upper_triangle))
            ]
        else:
            pairwise_features = [0.0, 0.0, 0.0]

        candidate_distances = distances[:, facility_idx]
        peer_distances = np.linalg.norm(
            facility_coords - facility_coords[facility_idx], axis=1
        )
        peer_distances = np.delete(peer_distances, facility_idx)
        mean_peer_distance = float(np.mean(peer_distances)) if peer_distances.size else 0.0

        features = [
            len(instance.demand_points),
            len(instance.facility_locations),
            instance.p,
            len(instance.demand_points) / len(instance.facility_locations),
            instance.p / len(instance.facility_locations),
            float(np.mean(flat_distances)),
            float(np.std(flat_distances)),
            float(np.min(flat_distances)),
            float(np.max(flat_distances)),
            float(np.median(flat_distances)),
            float(np.mean(demand_nearest)),
            float(np.std(demand_nearest)),
            float(np.max(demand_nearest)),
            float(np.mean(facility_coverage)),
            float(np.std(facility_coverage)),
            float(np.min(facility_coverage)),
            float(np.mean(np.linalg.norm(demand_coords - demand_center, axis=1))),
            float(np.mean(np.linalg.norm(facility_coords - facility_center, axis=1))),
            float(np.linalg.norm(demand_center - facility_center)),
            *pairwise_features,
            float(np.mean(candidate_distances)),
            float(np.std(candidate_distances)),
            float(np.max(candidate_distances)),
            mean_peer_distance
        ]
        return np.asarray(features, dtype=float)

    def get_feature_names(self) -> List[str]:
        """Return feature names in the same order as the feature vector."""
        return self.feature_names.copy()

In [ ]:
class EnsemblePCenterSolver:
    """Rank facility candidates with a weighted ensemble of supervised models."""

    def __init__(self) -> None:
        self.feature_extractor = FeatureExtractor()
        self.scaler = StandardScaler()
        self.gb_regressor = GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42
        )
        self.rf_classifier = RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=42
        )
        self.nn_regressor = MLPRegressor(
            hidden_layer_sizes=(64, 32), activation="relu",
            solver="adam", max_iter=500, random_state=42
        )
        self.models = {
            "gradient_boosting": self.gb_regressor,
            "random_forest": self.rf_classifier,
            "neural_network": self.nn_regressor
        }
        self.model_weights = {
            "gradient_boosting": 0.4,
            "random_forest": 0.3,
            "neural_network": 0.3
        }
        self.validation_metrics: Dict[str, float] = {}
        self.is_trained = False

    def generate_training_instances(
        self, n_instances: int = 200
    ) -> List[Tuple[PCenterInstance, Set[int], float]]:
        """Generate small instances and obtain reference solutions by enumeration."""
        if n_instances < 2:
            raise ValueError("At least two instances are needed for a train/validation split.")

        training_data = []
        for instance_number in range(n_instances):
            n_demand = random.randint(4, 8)
            n_facilities = random.randint(5, 8)
            p = random.randint(2, min(4, n_facilities - 1))

            # Reset NumPy's generator for each instance so its coordinates are reproducible.
            np.random.seed(instance_number)
            demand_points = [
                (np.random.uniform(0, 10), np.random.uniform(0, 10))
                for _ in range(n_demand)
            ]
            facility_locations = [
                (np.random.uniform(0, 10), np.random.uniform(0, 10))
                for _ in range(n_facilities)
            ]
            instance = PCenterInstance(demand_points, facility_locations, p)
            optimal_solution, optimal_objective = instance.get_optimal_solution()
            training_data.append((instance, optimal_solution, optimal_objective))
        return training_data

    def prepare_training_data(
        self,
        training_instances: List[Tuple[PCenterInstance, Set[int], float]]
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Create one candidate-level feature row and binary label per facility."""
        X = []
        y_reg = []
        y_clf = []

        for instance, optimal_solution, _ in training_instances:
            for facility_idx in range(len(instance.facility_locations)):
                X.append(self.feature_extractor.extract_features(instance, facility_idx))
                selected = int(facility_idx in optimal_solution)
                y_reg.append(float(selected))
                y_clf.append(selected)

        return np.asarray(X), np.asarray(y_reg), np.asarray(y_clf, dtype=int)

    def train(self, n_instances: int = 200) -> None:
        """Train the three models and report candidate-label validation accuracy."""
        print("Generating training data...")
        training_instances = self.generate_training_instances(n_instances)
        train_instances, validation_instances = train_test_split(
            training_instances, test_size=0.2, random_state=42
        )

        X_train, y_reg_train, y_clf_train = self.prepare_training_data(train_instances)
        X_validation, y_reg_validation, y_clf_validation = self.prepare_training_data(
            validation_instances
        )
        print(f"Training data shape: {X_train.shape}")
        print(f"Feature dimension: {X_train.shape[1]}")

        # Fit scaling only on training rows so held-out instances do not affect preprocessing.
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_validation_scaled = self.scaler.transform(X_validation)

        print("Training Gradient Boosting Regressor...")
        self.gb_regressor.fit(X_train_scaled, y_reg_train)
        gb_predictions = np.clip(self.gb_regressor.predict(X_validation_scaled), 0.0, 1.0)
        gb_accuracy = float(np.mean((gb_predictions >= 0.5) == y_clf_validation))
        print(f"Gradient Boosting candidate-label accuracy: {gb_accuracy:.3f}")

        print("Training Random Forest Classifier...")
        self.rf_classifier.fit(X_train_scaled, y_clf_train)
        rf_predictions = self.rf_classifier.predict(X_validation_scaled)
        rf_accuracy = float(np.mean(rf_predictions == y_clf_validation))
        print(f"Random Forest candidate-label accuracy: {rf_accuracy:.3f}")

        print("Training Neural Network Regressor...")
        self.nn_regressor.fit(X_train_scaled, y_reg_train)
        nn_predictions = np.clip(self.nn_regressor.predict(X_validation_scaled), 0.0, 1.0)
        nn_accuracy = float(np.mean((nn_predictions >= 0.5) == y_clf_validation))
        print(f"Neural Network candidate-label accuracy: {nn_accuracy:.3f}")

        self.validation_metrics = {
            "Gradient Boosting": gb_accuracy,
            "Random Forest": rf_accuracy,
            "Neural Network": nn_accuracy
        }
        self.is_trained = True
        print("Ensemble training completed.")

    def predict_facility_scores(self, instance: PCenterInstance) -> np.ndarray:
        """Return normalized weighted model scores for all candidates."""
        if not self.is_trained:
            raise ValueError("Train the ensemble before requesting predictions.")

        candidate_features = np.asarray([
            self.feature_extractor.extract_features(instance, facility_idx)
            for facility_idx in range(len(instance.facility_locations))
        ])
        features_scaled = self.scaler.transform(candidate_features)

        gb_scores = np.clip(self.gb_regressor.predict(features_scaled), 0.0, 1.0)
        nn_scores = np.clip(self.nn_regressor.predict(features_scaled), 0.0, 1.0)
        rf_probabilities = self.rf_classifier.predict_proba(features_scaled)
        positive_class = np.flatnonzero(self.rf_classifier.classes_ == 1)
        if positive_class.size:
            rf_scores = rf_probabilities[:, positive_class[0]]
        else:
            rf_scores = np.zeros(len(instance.facility_locations))

        scores = (
            self.model_weights["gradient_boosting"] * gb_scores
            + self.model_weights["random_forest"] * rf_scores
            + self.model_weights["neural_network"] * nn_scores
        )
        score_total = float(np.sum(scores))
        if score_total > 0.0:
            scores = scores / score_total
        else:
            scores = np.full(len(instance.facility_locations), 1.0 / len(instance.facility_locations))
        return scores

    def solve(self, instance: PCenterInstance) -> Tuple[Set[int], float, np.ndarray]:
        """Select the top p scored candidates and evaluate their p-center radius."""
        if not self.is_trained:
            raise ValueError("Train the ensemble before solving an instance.")

        facility_scores = self.predict_facility_scores(instance)
        top_facilities = np.argsort(facility_scores, kind="stable")[-instance.p:]
        predicted_solution = set(int(index) for index in top_facilities)
        predicted_objective = instance.evaluate_solution(predicted_solution)
        return predicted_solution, predicted_objective, facility_scores

In [ ]:
# Define the fixed example so its coordinates and p-value are visible before training.
print("Ensemble Learning for the p-Center Problem")
print("=" * 44)

demand_points = [(1, 1), (2, 3), (4, 2), (5, 4), (3, 1)]
facility_locations = [(1.5, 1.5), (3, 2), (4.5, 3), (2, 4)]
p = 2
test_instance = PCenterInstance(demand_points, facility_locations, p)

print("Test problem:")
print(f"Demand points: {demand_points}")
print(f"Facility candidates: {facility_locations}")
print(f"p = {p}")

# Exhaustive enumeration supplies the reference objective for this small instance.
optimal_solution, optimal_objective = test_instance.get_optimal_solution()
print(f"Optimal facility indices: {sorted(optimal_solution)}")
print(f"Optimal objective: {optimal_objective:.3f}")

In [ ]:
ensemble = EnsemblePCenterSolver()
ensemble.train(n_instances=200)

predicted_solution, predicted_objective, facility_scores = ensemble.solve(test_instance)
if optimal_objective > 0:
    optimality_gap = 100.0 * (predicted_objective - optimal_objective) / optimal_objective
else:
    optimality_gap = 0.0 if math.isclose(predicted_objective, 0.0, abs_tol=1e-12) else float("inf")

print("\nEnsemble prediction:")
print(f"Selected facility indices: {sorted(predicted_solution)}")
print(f"Predicted objective: {predicted_objective:.3f}")
print(f"Normalized facility scores: {facility_scores.round(3)}")
print("\nComparison with exhaustive enumeration:")
print(f"Optimal objective: {optimal_objective:.3f}")
print(f"Ensemble objective: {predicted_objective:.3f}")
print(f"Objective gap: {optimality_gap:.2f}%")
print(
    "Objective matches the optimum: "
    f"{math.isclose(predicted_objective, optimal_objective, rel_tol=1e-9, abs_tol=1e-12)}"
)

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))

# The first panel shows demand, selected candidates, and nearest-facility assignments.
ax1.set_title("Ensemble Facility Selection")
for demand_idx, (x_coord, y_coord) in enumerate(demand_points):
    ax1.scatter(x_coord, y_coord, s=100, c="tab:red", marker="s", label="Demand point" if demand_idx == 0 else None)
    ax1.annotate(f"D{demand_idx + 1}", (x_coord, y_coord), xytext=(4, 4), textcoords="offset points")

for facility_idx, (x_coord, y_coord) in enumerate(facility_locations):
    selected = facility_idx in predicted_solution
    ax1.scatter(
        x_coord, y_coord, s=180, c="tab:green" if selected else "lightgray",
        marker="^" if selected else "o", label="Selected facility" if selected and facility_idx == min(predicted_solution) else None
    )
    ax1.annotate(
        f"F{facility_idx + 1} ({facility_scores[facility_idx]:.2f})",
        (x_coord, y_coord), xytext=(4, 4), textcoords="offset points"
    )

distances = test_instance.compute_distance_matrix()
for demand_idx, (x_coord, y_coord) in enumerate(demand_points):
    nearest_facility = min(predicted_solution, key=lambda idx: distances[demand_idx, idx])
    facility_x, facility_y = facility_locations[nearest_facility]
    ax1.plot([x_coord, facility_x], [y_coord, facility_y], "k--", alpha=0.35, linewidth=1)
ax1.set_xlabel("X coordinate (coordinate units)")
ax1.set_ylabel("Y coordinate (coordinate units)")
ax1.set_xlim(-1, 6)
ax1.set_ylim(0, 5)
ax1.axis("equal")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Scores are ranking values, not calibrated probabilities of optimal selection.
facility_labels = [f"F{idx + 1}" for idx in range(len(facility_locations))]
bar_colors = ["tab:green" if idx in predicted_solution else "lightgray" for idx in range(len(facility_locations))]
ax2.bar(facility_labels, facility_scores, color=bar_colors, alpha=0.8)
ax2.set_title("Weighted Ensemble Candidate Scores")
ax2.set_ylabel("Normalized score (unitless)")
ax2.set_ylim(0, max(facility_scores) * 1.2)
ax2.grid(True, alpha=0.3, axis="y")

# Random Forest feature importance is a fitted-model diagnostic, not a causal effect.
feature_names = ensemble.feature_extractor.get_feature_names()
forest_importance = ensemble.rf_classifier.feature_importances_
top_indices = np.argsort(forest_importance)[-10:]
ax3.barh(np.asarray(feature_names)[top_indices], forest_importance[top_indices], color="tab:blue", alpha=0.8)
ax3.set_title("Top Random Forest Feature Importances")
ax3.set_xlabel("Impurity-based importance (unitless)")
ax3.grid(True, alpha=0.3, axis="x")

# Compare the actual nearest-service distances with the worst-case objective.
nearest_distances = np.min(distances[:, sorted(predicted_solution)], axis=1)
ax4.bar([f"D{idx + 1}" for idx in range(len(demand_points))], nearest_distances, color="tab:purple", alpha=0.75)
ax4.axhline(predicted_objective, color="tab:red", linestyle="--", label="Maximum service distance")
ax4.set_title("Nearest-Selected-Facility Distance by Demand Point")
ax4.set_xlabel("Demand point")
ax4.set_ylabel("Distance (coordinate units)")
ax4.grid(True, alpha=0.3, axis="y")
ax4.legend()

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- The selected candidates are the p highest-scoring facilities.")
print("- Dashed segments connect each demand point to its nearest selected facility.")
print("- The largest displayed service distance equals the p-center objective, up to floating-point precision.")

In [ ]:
def test_ensemble_performance(n_test_instances: int = 20) -> List[Dict[str, float]]:
    """Evaluate objective gaps on independently generated small test instances."""
    if not ensemble.is_trained:
        raise ValueError("Train the ensemble before running the test experiment.")
    if n_test_instances < 1:
        raise ValueError("At least one test instance is required.")

    results = []
    for instance_number in range(n_test_instances):
        n_demand = random.randint(4, 7)
        n_facilities = random.randint(5, 8)
        p_value = random.randint(2, min(3, n_facilities - 1))
        demand = [
            (np.random.uniform(0, 10), np.random.uniform(0, 10))
            for _ in range(n_demand)
        ]
        facilities = [
            (np.random.uniform(0, 10), np.random.uniform(0, 10))
            for _ in range(n_facilities)
        ]
        instance = PCenterInstance(demand, facilities, p_value)
        _, exact_objective = instance.get_optimal_solution()
        selected, ensemble_objective, _ = ensemble.solve(instance)

        if exact_objective > 0:
            gap = 100.0 * (ensemble_objective - exact_objective) / exact_objective
        else:
            gap = 0.0 if math.isclose(ensemble_objective, 0.0, abs_tol=1e-12) else float("inf")

        results.append({
            "instance": instance_number + 1,
            "n_demand": n_demand,
            "n_facilities": n_facilities,
            "p": p_value,
            "optimal_objective": exact_objective,
            "ensemble_objective": ensemble_objective,
            "optimality_gap_percent": gap,
            "objective_matches_optimum": float(
                math.isclose(ensemble_objective, exact_objective, rel_tol=1e-9, abs_tol=1e-12)
            ),
            "selected_count": len(selected)
        })
    return results


test_results = test_ensemble_performance(20)
test_df = pd.DataFrame(test_results)
finite_gaps = test_df["optimality_gap_percent"].replace([np.inf, -np.inf], np.nan).dropna()

print("Evaluation on independently generated instances:")
print(f"Instances evaluated: {len(test_df)}")
print(f"Mean finite objective gap: {finite_gaps.mean():.2f}%")
print(f"Standard deviation of finite gaps: {finite_gaps.std():.2f}%")
print(f"Largest finite objective gap: {finite_gaps.max():.2f}%")
print(f"Objective-match rate: {test_df['objective_matches_optimum'].mean():.1%}")
print("These small-instance results measure this generated sample; they do not establish performance on other distributions.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(finite_gaps, bins=15, color="tab:blue", alpha=0.75, edgecolor="black")
ax1.axvline(finite_gaps.mean(), color="tab:red", linestyle="--", label=f"Mean: {finite_gaps.mean():.2f}%")
ax1.set_xlabel("Objective gap (%)")
ax1.set_ylabel("Number of instances")
ax1.set_title("Ensemble Objective-Gap Distribution")
ax1.grid(True, alpha=0.3)
ax1.legend()

scatter = ax2.scatter(
    test_df["n_demand"], test_df["optimality_gap_percent"],
    c=test_df["p"], cmap="viridis", s=70, alpha=0.8
)
ax2.set_xlabel("Number of demand points")
ax2.set_ylabel("Objective gap (%)")
ax2.set_title("Objective Gap Across Test Instances")
ax2.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax2, label="Number of facilities p")
plt.tight_layout()
plt.show()

In [ ]:
# These are component-level validation diagnostics, not competing p-center solvers.
model_comparison = pd.DataFrame(
    {
        "Model": list(ensemble.validation_metrics.keys()),
        "Candidate-label accuracy": list(ensemble.validation_metrics.values())
    }
)

print("Held-out candidate-label diagnostics:")
print(model_comparison.to_string(index=False, formatters={"Candidate-label accuracy": "{:.3f}".format}))
print(
    "Validation rows come from held-out problem instances. Accuracy measures candidate selection labels; "
    "it does not directly measure the quality of the final p-facility set."
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    model_comparison["Model"],
    model_comparison["Candidate-label accuracy"],
    color=["tab:blue", "tab:orange", "tab:green"],
    alpha=0.8
)
ax.set_ylabel("Candidate-label accuracy")
ax.set_ylim(0, 1)
ax.set_title("Ensemble Component Validation Diagnostics")
ax.grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, model_comparison["Candidate-label accuracy"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.3f}", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the candidate-level features for the concrete example.
feature_names = ensemble.feature_extractor.get_feature_names()
candidate_feature_matrix = np.asarray([
    ensemble.feature_extractor.extract_features(test_instance, facility_idx)
    for facility_idx in range(len(facility_locations))
])
if candidate_feature_matrix.shape[1] != len(feature_names):
    raise AssertionError("Feature names and extracted feature values must have matching dimensions.")

print("Candidate feature matrix:")
print(f"Shape: {candidate_feature_matrix.shape}")
print("Each row corresponds to one candidate; columns follow the listed feature order.")
print(pd.DataFrame(candidate_feature_matrix, columns=feature_names).round(3).to_string(index=False))

# Show absolute feature values for one candidate without interpreting them as importance.
example_candidate = int(np.argmax(facility_scores))
example_values = candidate_feature_matrix[example_candidate]
top_value_indices = np.argsort(np.abs(example_values))[-10:]
print(f"\nLargest-magnitude feature values for candidate F{example_candidate + 1}:")
for feature_idx in top_value_indices[::-1]:
    print(f"{feature_names[feature_idx]:35s} {example_values[feature_idx]:9.3f}")
print("Feature magnitude alone does not measure a feature's predictive contribution.")

# Group fitted forest importances by the feature categories used in this model.
categories = {
    "Size and ratios": feature_names[:5],
    "Distance summaries": feature_names[5:10],
    "Coverage summaries": feature_names[10:16],
    "Spatial and candidate summaries": feature_names[16:]
}
forest_importance = ensemble.rf_classifier.feature_importances_
category_importance = {
    category: float(sum(forest_importance[feature_names.index(name)] for name in names))
    for category, names in categories.items()
}

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))
top_features = np.argsort(forest_importance)[-12:]
ax1.barh(np.asarray(feature_names)[top_features], forest_importance[top_features], color="tab:blue", alpha=0.8)
ax1.set_xlabel("Random Forest impurity-based importance")
ax1.set_title("Most Important Features in the Fitted Forest")
ax1.grid(True, alpha=0.3, axis="x")

ax2.bar(category_importance.keys(), category_importance.values(), color="tab:orange", alpha=0.8)
ax2.set_ylabel("Summed Random Forest importance")
ax2.set_title("Forest Importance by Feature Category")
ax2.tick_params(axis="x", rotation=25)
ax2.grid(True, alpha=0.3, axis="y")

key_features = ["n_demand", "p", "mean_distance", "max_demand_nearest", "demand_spread"]
key_indices = [feature_names.index(name) for name in key_features]
key_values = candidate_feature_matrix[example_candidate, key_indices]
if np.std(key_values) > 0:
    correlation = np.corrcoef(candidate_feature_matrix[:, key_indices], rowvar=False)
else:
    correlation = np.zeros((len(key_indices), len(key_indices)))
    np.fill_diagonal(correlation, 1.0)
image = ax3.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax3.set_xticks(range(len(key_features)), key_features, rotation=40, ha="right")
ax3.set_yticks(range(len(key_features)), key_features)
ax3.set_title("Correlation Across Candidate Rows in This Instance")
fig.colorbar(image, ax=ax3, label="Pearson correlation")

ax4.bar(facility_labels, facility_scores, color=bar_colors, alpha=0.8)
ax4.set_ylabel("Normalized ensemble score")
ax4.set_title("Candidate Scores (Ranking Values, Not Confidence Estimates)")
ax4.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()
print("Random Forest importance is a model-specific diagnostic and should not be read as causal evidence.")

## Interpreting the method

### Strengths and limitations

The ensemble reuses labeled small instances to learn a candidate-ranking rule. Once trained, it produces scores for a new instance without enumerating every \(p\)-facility set. The score calculation is therefore based on evaluating the candidate-level features and the three fitted models, followed by selecting the top \(p\) scores.

This speed-oriented prediction does not provide an optimality guarantee. Training quality depends on the generated instances, the feature representation, the selected weights, and the models' ability to generalize. Candidate-level validation accuracy is not a substitute for measuring the final p-center objective; the independent-instance experiment evaluates that objective directly. The exhaustive reference remains practical here because the instances are small.

The scores are normalized ranking values, not calibrated probabilities or uncertainty estimates. The Random Forest feature importances describe how that fitted model uses features under its impurity-based criterion; they do not establish causal relationships. Tied optimal solutions can also create ambiguous training labels because the enumeration stores one optimal set.

### Appropriate use

This approach is most relevant when similar small instances can provide labeled examples and rapid candidate ranking is useful. Before applying it to operational decisions, assess objective gaps on representative held-out instances and verify that the training distribution reflects the intended setting. Larger instances, additional operational constraints, or substantial distribution shifts require further modeling and validation; none are implemented in this example.